# population

In [ ]:
#| default_exp game/settlement

In [ ]:
#| export
from fastcore.basics import patch

# fun with colors
import colorsys
import seaborn as sns
import matplotlib.pyplot as plt
from importlib import resources
import pandas as pd
import random

In [ ]:
#| export
from HexMagic.styles import   SVGBuilder, SVGDef,  Generatable, NamedColor, StyleCSS
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord , Hex, HexGrid, PrimitiveDemo
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion ,  unique_windy_edge
from HexMagic.terrainpatterns import TerrainPatterns, PathPattern
from HexMagic.styles import apply_looping_animation, LoopingLayerAnimation
from HexMagic.terrain import Terrain


In [ ]:
#| export
class CountryFlag:

    def __init__(self, c, name, year):
        self.primary = plt.matplotlib.colors.rgb2hex(c)
        h, s, v = colorsys.rgb_to_hsv(*c)
        self.comp = plt.matplotlib.colors.rgb2hex(colorsys.hsv_to_rgb((h + 0.5) % 1, s, v))
        self.tri1 = plt.matplotlib.colors.rgb2hex(colorsys.hsv_to_rgb((h + 1/3) % 1, s, v))
        self.tri2 = plt.matplotlib.colors.rgb2hex(colorsys.hsv_to_rgb((h - 1/3) % 1, s, v))
        self.name = name
        self.year = year
        self.countryPrefix = CountryFlag.countryName(name)
        self.capital = CountryFlag.cityName(name)

    @classmethod
    def seaborn(cls,name:str,levels=7,gender=None,year=1900):
        palette = sns.color_palette(name, levels)

        if gender is not None:
            names = random.sample(CountryFlag.commonNames(year=year,gender=gender),levels)
        else:
            allN = CountryFlag.commonNames(year=year,gender="M") + CountryFlag.commonNames(year=year,gender="F")
            names = random.sample(allN,levels)
            
       
        ret = []
        for i, color in enumerate(palette):
            ret.append(cls(color,name=names[i],year=year))
        return ret

    
    @classmethod
    def commonNames(cls,year=1900,gender="M"):
        
        with resources.files('HexMagic').joinpath('data/misc/popularNames.csv').open() as f:
            df = pd.read_csv(f)
        
        filtered = df[(df['Year'] == year) & (df['Gender'] == gender)]
        if filtered.empty:
            raise ValueError(f"No names found for year={year}, gender={gender}")
        
        # Weight by count for more realistic distribution
        return filtered['Name'].tolist()


    @classmethod
    def countryName(cls, name , pattern=None, descriptor=None):
        # Dictionary of place descriptors organized by first letter
        PLACE_DESCRIPTORS = {
            'A': ['Abbey', 'Acres', 'Alcove', 'Apex', 'Archipelago', 'Arena', 'Atoll', 'Avenue'],
            'B': ['Basin', 'Bay', 'Bluff', 'Borough', 'Boundary', 'Bower', 'Burg', 'Borderlands'],
            'C': ['Canyon', 'Cape', 'Castle', 'Citadel', 'Clearing', 'Cove', 'Crossing', 'County'],
            'D': ['Dale', 'Dell', 'Delta', 'Den', 'District', 'Domain', 'Dunes', 'Dominion'],
            'E': ['Edge', 'Enclave', 'End', 'Estate', 'Expanse', 'Empire', 'Escarpment', 'Eyrie'],
            'F': ['Falls', 'Fen', 'Fjord', 'Forest', 'Fort', 'Frontier', 'Fields', 'Fief'],
            'G': ['Gap', 'Garden', 'Gate', 'Glade', 'Glen', 'Gorge', 'Grotto', 'Grove'],
            'H': ['Habitat', 'Harbor', 'Haven', 'Heath', 'Heights', 'Hideaway', 'Hill', 'Hollow'],
            'I': ['Isle', 'Inlet', 'Island', 'Isthmus', 'Ironworks', 'Imperium', 'Inn', 'Impasse'],
            'J': ['Junction', 'Jungle', 'Jetty', 'Juncture', 'Jurisdiction', 'Jut', 'Joint', 'Jewel'],
            'K': ['Keep', 'Kingdom', 'Knoll', 'Key', 'Knot', 'Kiosk', 'Krantz', 'Karst'],
            'L': ['Lagoon', 'Lake', 'Landing', 'Land', 'Lair', 'Ledge', 'Lodge', 'Lowlands'],
            'M': ['Manor', 'Marsh', 'Meadow', 'Mesa', 'Moor', 'Mount', 'Mountains', 'Mound'],
            'N': ['Narrows', 'Nest', 'Niche', 'Nook', 'North', 'Notch', 'Nation', 'Neighborhood'],
            'O': ['Oasis', 'Observatory', 'Outpost', 'Overlook', 'Orchard', 'Outcrop', 'Outlet', 'Outlands'],
            'P': ['Palace', 'Park', 'Pass', 'Path', 'Peak', 'Peninsula', 'Pinnacle', 'Plaza', 'Point', 'Province'],
            'Q': ['Quarry', 'Quarter', 'Quay', 'Quarters', 'Quad', 'Quadrant', 'Quest', 'Quietude'],
            'R': ['Range', 'Ravine', 'Reach', 'Realm', 'Reef', 'Region', 'Reserve', 'Retreat', 'Ridge', 'Rise'],
            'S': ['Sanctuary', 'Settlement', 'Shire', 'Shore', 'Slopes', 'Sound', 'Span', 'Spring', 'Summit', 'Stronghold'],
            'T': ['Terrace', 'Territory', 'Thicket', 'Timberland', 'Tower', 'Town', 'Trail', 'Trench', 'Tundra', 'Township'],
            'U': ['Undergrowth', 'Underpass', 'Union', 'Uplands', 'Upper', 'Utopia', 'Utterness', 'Umbrage'],
            'V': ['Vale', 'Valley', 'Vault', 'View', 'Villa', 'Village', 'Vineyards', 'Vista', 'Void', 'Vanguard'],
            'W': ['Ward', 'Wasteland', 'Water', 'Way', 'Wetlands', 'Wilds', 'Wood', 'Woods', 'Works', 'Warren'],
            'X': ['Xanadu', 'Xenolith', 'Xerophyte', 'X-Roads', 'Xeric', 'Xyst', 'X-Point', 'X-ing'],
            'Y': ['Yard', 'Yonder', 'Yurt', 'Yards', 'Yielding', 'York', 'Yukon', 'Yews'],
            'Z': ['Zone', 'Zenith', 'Zephyr', 'Zigzag', 'Ziggurat', 'Zion', 'Zodiac', 'Zocalo'],
        }

        # Possessive patterns
        PATTERNS = [
            "{name}'s {place}",      # Karl's Kingdom
            "{place} of {name}",     # Kingdom of Karl
            "{name} {place}",        # Karl Kingdom
        ]

        if not name:
            return ""
        
        first_letter = name[0].upper()
        
        # Get possible descriptors for this letter
        descriptors = PLACE_DESCRIPTORS.get(first_letter, ['Place', 'Point', 'Precinct'])
        
        # Choose descriptor
        if descriptor and descriptor in descriptors:
            place = descriptor
        else:
            place = random.choice(descriptors)

        return place
        
        # Choose pattern
        if pattern is not None and 0 <= pattern < len(PATTERNS):
            template = PATTERNS[pattern]
        else:
            template = random.choice(PATTERNS)
        
        return template.format(name=name, place=place)

    @classmethod
    def cityName(cls, name , pattern=None, descriptor=None, use_suffix=None):
        # Dictionary of place descriptors organized by first letter
        SETTLEMENT_DESCRIPTORS = {
            'A': ['Acres', 'Arbor', 'Ashton', 'Auburn', 'Avon', 'Aldridge', 'Ashford', 'Aston'],
            'B': ['Bay', 'Beach', 'Bridge', 'Brook', 'Burg', 'Borough', 'Bluff', 'Bend'],
            'C': ['City', 'Cove', 'Creek', 'Crest', 'Crossing', 'Center', 'Cape', 'Corners'],
            'D': ['Dale', 'Dell', 'Dunes', 'Down', 'Dock', 'Delta', 'Downs', 'Den'],
            'E': ['End', 'Edge', 'Estates', 'Elms', 'Enclave', 'Evergreen', 'East', 'Elm'],
            'F': ['Falls', 'Field', 'Fields', 'Ford', 'Forest', 'Fort', 'Forks', 'Ferry'],
            'G': ['Glen', 'Glade', 'Green', 'Grove', 'Gate', 'Gardens', 'Groves', 'Gap'],
            'H': ['Harbor', 'Haven', 'Heights', 'Hill', 'Hills', 'Hollow', 'Heath', 'Hurst'],
            'I': ['Isle', 'Island', 'Inlet', 'Inn', 'Ironworks', 'Ivy', 'Isles', 'Inches'],
            'J': ['Junction', 'Jetty', 'Juncture', 'Junction', 'Jamestown', 'Jardin', 'Jct', 'Joya'],
            'K': ['Key', 'Knoll', 'Knolls', 'Keep', 'Keystone', 'Kingswood', 'Kirk', 'Knolle'],
            'L': ['Lake', 'Landing', 'Lawn', 'Ledge', 'Lock', 'Lodge', 'Lagoon', 'Lynn'],
            'M': ['Manor', 'Meadow', 'Meadows', 'Mill', 'Mills', 'Mount', 'Moor', 'Mountain'],
            'N': ['North', 'Nook', 'Narrows', 'Neck', 'Nest', 'Newton', 'New', 'Notch'],
            'O': ['Oaks', 'Orchard', 'Overlook', 'Outpost', 'Outlet', 'Oak', 'Oasis', 'Old'],
            'P': ['Park', 'Pines', 'Plains', 'Point', 'Pond', 'Port', 'Plaza', 'Pass'],
            'Q': ['Quarry', 'Quarter', 'Quay', 'Queen', 'Quarters', 'Quayside', 'Quest', 'Quince'],
            'R': ['Ridge', 'River', 'Rock', 'Run', 'Ranch', 'Rapids', 'Reach', 'Rest'],
            'S': ['Springs', 'Shore', 'Shores', 'South', 'Station', 'Summit', 'Shire', 'Side'],
            'T': ['Town', 'Terrace', 'Trace', 'Trail', 'Township', 'Tower', 'Thicket', 'Timber'],
            'U': ['Union', 'Uplands', 'Upper', 'Underwood', 'Unity', 'University', 'Upton', 'Utopia'],
            'V': ['Vale', 'Valley', 'View', 'Villa', 'Village', 'Vista', 'Ville', 'Vineyards'],
            'W': ['West', 'Water', 'Waters', 'Way', 'Wells', 'Wood', 'Woods', 'Wick'],
            'X': ['Xanadu', 'X-Roads', 'Xing', 'Xavier', 'Xeric', 'Xenia', 'Xenophon', 'Xyst'],
            'Y': ['Yard', 'Yonder', 'York', 'Yards', 'Yew', 'Yews', 'Yale', 'Yarmouth'],
            'Z': ['Zone', 'Zenith', 'Zephyr', 'Zion', 'Zinc', 'Zodiac', 'Zona', 'Zuni'],
        }

        # Naming patterns for settlements
        PATTERNS = [
            "{name}ville",           # Karlville
            "{name}ton",             # Karlton
            "{name}burg",            # Karlburg
            "{name}wood",            # Karlwood
            "{name} {place}",        # Karl Creek
            "{name}'s {place}",      # Karl's Crossing
            "{place} of {name}",     # City of Karl
            "New {name}",            # New Karl
            "Old {name}",            # Old Karl
            "Little {name}",         # Little Karl
            "Upper {name}",          # Upper Karl
            "Lower {name}",          # Lower Karl
            "East {name}",           # East Karl
            "West {name}",           # West Karl
            "North {name}",          # North Karl
            "South {name}",          # South Karl
        ]

        # Shorter patterns for descriptors (avoid double suffixes)
        DESCRIPTOR_PATTERNS = [
            "{name} {place}",        # Karl Creek
            "{name}'s {place}",      # Karl's Crossing
            "{place} of {name}",     # City of Karl
        ]

        if not name:
            return ""
        
        first_letter = name[0].upper()
        
        # Get possible descriptors for this letter
        descriptors = SETTLEMENT_DESCRIPTORS.get(first_letter, ['Place', 'Point', 'Plaza'])
        
        # Decide whether to use suffix or descriptor pattern
        if use_suffix is None:
            use_suffix = random.choice([True, False])
        
        if use_suffix:
            # Use simple suffix patterns (first 7 patterns)
            patterns = PATTERNS[:7]
            if pattern is not None and 0 <= pattern < len(patterns):
                template = patterns[pattern]
            else:
                template = random.choice(patterns)
            
            if "{place}" in template:
                place = descriptor if descriptor and descriptor in descriptors else random.choice(descriptors)
                return template.format(name=name, place=place)
            else:
                return template.format(name=name)
        else:
            # Use directional/size prefix patterns (last 9 patterns)
            patterns = PATTERNS[7:]
            if pattern is not None and 0 <= pattern < len(patterns):
                template = patterns[pattern]
            else:
                template = random.choice(patterns)
            return template.format(name=name)

    @staticmethod
    def decode(s: str) -> 'CountryFlag':
        """Decode CountryFlag from string."""
        parts = s.split('|')
        primary = parts[0]
        name = parts[1]
        year = int(parts[2])
        
        # Convert hex back to RGB tuple (0-1 range)
        rgb = plt.matplotlib.colors.to_rgb(primary)
        
        return CountryFlag(rgb, name=name, year=year)

In [ ]:
#| export
@patch
def encode(self: CountryFlag) -> str:
    """Encode CountryFlag to a single line string."""
    # Format: primary|name|year
    return f"{self.primary}|{self.name}|{self.year}"

In [ ]:
#| export
@patch
def plain(self:CountryFlag,name,width=3):
    return StyleCSS(name,fill=self.primary,stroke=self.comp,stroke_width=width)

@patch
def kingStyle(self:CountryFlag,name,width=2):
    #saturation = 0.7
    style = self.plain(name,width)
    style.name = f"country_{name}"
    #style.properties["fill"] = style.desaturate(saturation).properties["fill"]
    style.properties["opacity"] = 0.7
    return style

@patch
def contrastStyle(self:CountryFlag,name,width=1.5):
    return StyleCSS(
            f"contrast_{name}",
            fill=self.comp,
            stroke="#000",
            stroke_width=width
        )

@patch
def labelStyle(self:CountryFlag,name,width=1):
    return StyleCSS(
            f"contrast_{name}",
            fill=self.tri1,
            stroke="#36454F",
            stroke_width=width
        )

In [ ]:
flags = CountryFlag.seaborn("husl",4)
for flag in flags:
    print(flag.name , flag.capital)


In [ ]:
hexCount = 9
radius = 30
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    style = flat.plain(f"flag_{i}",width=10)
    sampleHex = Hex(radius=30, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

In [ ]:
#| export


@patch
def circlePattern(self:CountryFlag, id):
        """Generate a circle pattern definition"""
        content = f'<rect width=120 height= 90 fill="{self.comp}"/><circle cx="{50}" cy="{45}" r="{30}" fill="{self.primary}"/>'
        return SVGDef("pattern", id, content, 
                    width=120, height=90, 
                    patternUnits="userSpaceOnUse")


In [ ]:
#| export
@patch
def triPattern(self:CountryFlag, id):
        """Generate a circle pattern definition"""
        content = f'''<rect width=96 height= 96 fill="{self.primary}"/>
        <circle cx="{48}" cy="{48}" r="{48}" fill="{self.comp}"/>
        <circle cx="{48}" cy="{48}" r="{32}" fill="{self.tri1}"/>
        <circle cx="{48}" cy="{48}" r="{16}" fill="{self.tri2}"/>'''
        
        return SVGDef("pattern", id, content, 
                    width=96, height=96, 
                    patternUnits="userSpaceOnUse")

In [ ]:
#| export
@patch
def swirl(self:CountryFlag,id):

    content = f"""<g  fill='{self.primary}'><rect width=400 height=400 fill='{self.primary}' /></g>
<g  fill='{self.comp}' fill-opacity='1'><path d='M400 58.58c-38.95 0-74.21 15.74-99.79 41.21c-25.61 25.72-61.05 41.64-100.21 41.64s-74.6-15.92-100.21-41.64C74.21 74.32 38.95 58.58 0 58.58c-78.11 0-141.42 63.32-141.42 141.42S-78.11 341.42 0 341.42c38.95 0 74.21-15.74 99.79-41.21c25.61-25.72 61.05-41.64 100.21-41.64s74.6 15.92 100.21 41.64c25.58 25.47 60.84 41.21 99.79 41.21c78.11 0 141.42-63.32 141.42-141.42S478.11 58.58 400 58.58z'/><circle  cx='200' cy='1' r='60'/><circle  cx='200' cy='400' r='60'/></g><g  fill='{self.primary}'><circle  cx='0' cy='200' r='60'/><circle  cx='400' cy='200' r='60'/></g>
"""
    pat = SVGDef("pattern", id, content,
                  width=400, height=400,
                  patternUnits="userSpaceOnUse")
    scale = 0.2
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat

In [ ]:
hexCount = 9
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.swirl(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

In [ ]:
@patch
def yin(self:CountryFlag,id):
    content = f"""<rect fill='{self.primary}' width='100' height='100'/>
     	<g id="YinYang" fill="#000" stroke="none" stroke-width="0" fill-rule="evenodd">
		<title>Yin-Yang, by Adam Stanislav</title>
		<desc>The entire graphic is drawn as a single path filled with black (or any other color you change the value of “fill” in line 4). The other half, usually shown in white is created here as a hole in the path. That means it is completely transparent, and has whatever color its background has. To achieve this, not just with SVG but with other vector formats, any black portion of the path is drawn counterclockwise, any “white” portion clockwise. Also, this graphic is taking advantage of the kappa constant described in my e-book Bézier Circles and other shapes, freely downloadable from https://www.smashwords.com/books/view/483578 .</desc>

		<!-- Note to self: Relative Bézier differences (“c”) are differences of a point from the STARTING point of the curve segment, not from the most recent point. -->
		<path d="M400 0C179.086 0 0 179.086 0 400 0 620.914 179.086 800 400 800 620.914 800 800 620.914 800 400 800 179.086 620.914 0 400 0zM400 10C184.609 10 10 184.609 10 400 10 615.391 184.609 790 400 790 292.304 790 205 682.304 205 600 205 492.3 292.304 400 400 400 507.7 400 600 292.304 600 200 600 92.304 507.7 10 400 10zM400 665c35.895 0 65-29.105 65-65 0-35.895-29.105-65-65-65-35.895 0-65 29.105-65 65 0 35.895 29.105 65 65 65zM400 132c-37.555 0-68 30.445-68 68 0 37.555 30.445 68 68 68 37.555 0 68-30.445 68-68 0-37.555-30.445-68-68-68z"/>
	</g>
  """


    pat = SVGDef("pattern", id, content,
                  width=100, height=100,
                  patternUnits="userSpaceOnUse")
    scale = 0.5
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat

In [ ]:
hexCount = 9
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.yin(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

In [ ]:
hexCount = 9
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.triPattern(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

In [ ]:
for pat in canvas.definitions:
    print(pat.attributes["id"])

In [ ]:


hexStyles = CountryFlag.seaborn("Accent", hexCount)
hexIndex = 3
name = f"CountrySwirl_{hexIndex}"
patternName = f"{name}_pat"
hexPat = hexStyles[hexIndex].swirl(patternName)

#some drawing setup 
hexStyle = StyleCSS(name, fill=f"url(#{patternName})")
canvas = SVGBuilder()
canvas.width=200 ;canvas.height=200
canvas.add_style(hexStyle)
canvas.add_definition(hexPat)

# add our hex to the canvas
sampleHex = Hex(radius=50,center=MapCord(100,100),style=hexStyle)
canvas.adjust("main",sampleHex.svg())

#show our work
canvas.show()
canvas.adjust("main",sampleHex.svg())

#show our work
canvas.show()

